# Cluster Analysis

This notebook provides a detailed evaluation of 3D Gaussian Clustering results by leveraging visibility statistics rendered from training camera views.
It helps refine the semantic segmentation of the scene by identifying "meaningful" objects versus "visual noise" or artifacts.

In [1]:
import argparse
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm import tqdm

from opensplat3d.language.embed import compute_view_stats
from opensplat3d.language.utils import RenderParams
from opensplat3d.params import PipeParams
from opensplat3d.utils.setup_utils import setup

In [2]:
model_dir = Path("/home/jsm15/opensplat3d/outputs/lerf_ovs_waldo_kitchen/20260322124012-d4653873")
setup_params = setup(model_dir)

labels_path = model_dir / Path("clustering/labels.npy")

labels = torch.from_numpy(np.load(labels_path))
unique_labels = labels.unique()
unique_labels = unique_labels[unique_labels != -1]

cameras = setup_params.scene.get_train_cameras()

pipe_params = PipeParams()
bg_color = [1, 1, 1] if setup_params.model_params.white_background else [0, 0, 0]
bg = torch.tensor(bg_color, dtype=torch.float32, device=setup_params.device)

render_params = RenderParams(
    setup_params.gaussians,
    setup_params.model_params,
    cameras,
    pipe_params,
    bg,
)

Iteration 30000 | num_gaussians: 1106422 | feature_dim: 8
Detected lerf-ovs format.
Detected lerf-ovs format.


Loading training cameras:   0%|          | 0/182 [00:00<?, ?it/s]

Loading testing cameras:   0%|          | 0/5 [00:00<?, ?it/s]

Train: 182 | Test: 5
Sampled 17475/17475 points from the point cloud.
Loading Training Cameras


  0%|          | 0/182 [00:00<?, ?it/s]

Loading Test Cameras


  0%|          | 0/5 [00:00<?, ?it/s]

In [3]:
def plot_clusters(render_params, pred_threshold=0.2):

    appearances_list = []
    avg_scores_list = []
    cluster_ids = []

    for label in tqdm(unique_labels, desc="Evaluating clusters", total=unique_labels.shape[0]):
        label_id = int(label.item())
        label_mask = labels == label

        mask_color = torch.tensor([[1.0, 1.0, 1.0]], device=setup_params.device).repeat(
            labels.shape[0], 1
        )
        mask_color[~label_mask] = 0

        label_count = int(label_mask.sum().item())

        stats = compute_view_stats(
            render_params,
            mask_color,
            pred_threshold,
            label_mask,
            label_count,
        )

        max_area = max([x.area for x in stats]) if stats else 0

        appearances = 0
        total_score = 0.0

        if max_area > 0:
            for stat in stats:
                if stat.area > 0:
                    appearances += 1
                    score = (stat.visible_count / stat.label_count) * (stat.area / max_area)
                    total_score += score

        avg_score = total_score / appearances if appearances > 0 else 0.0

        appearances_list.append(appearances)
        avg_scores_list.append(avg_score)
        cluster_ids.append(str(label_id))

    return (appearances_list, avg_scores_list, cluster_ids)

In [5]:
appearances_list, avg_scores_list, cluster_ids = plot_clusters(render_params,0.2)

Evaluating clusters: 100%|██████████| 325/325 [08:43<00:00,  1.61s/it]


In [1]:
plt.figure(figsize=(10, 6))
plt.scatter(appearances_list, avg_scores_list, alpha=0.7, edgecolors='k')

plt.title("Cluster Visualization Quality")
plt.xlabel("Number of Appearances (Views with Area > 0)")
plt.ylabel("Average Visualization Score")
plt.grid(True, linestyle='--', alpha=0.6)
    
# Optional styling to help visualize common thresholds
plt.axvline(x=5, color='r', linestyle='--', alpha=0.4, label='x=5 (Typical Min Appearances)')
plt.axhline(y=0.05, color='orange', linestyle='--', alpha=0.4, label='y=0.05 (Typical Min Score)')
plt.legend()

plt.show()

NameError: name 'plt' is not defined